<a href="https://colab.research.google.com/github/NicEzWs2983/HARCNN_TSM/blob/main/Temporal_Shift_Module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import glob
import time
import torch
import random
import kagglehub
import numpy as np


from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torch.optim.lr_scheduler as lr_scheduler

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

print("Done") # 如果使用jupyter能夠更清楚知道這塊cell已經跑完了

Done


In [ ]:
# Connecting google drive for saving weight of model
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Dataset Processing**

## Download Dataset latest version

In [ ]:
path = kagglehub.dataset_download("easonlll/hmdb51") + "/HMDB51"

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'hmdb51' dataset.
Path to dataset files: /kaggle/input/hmdb51/HMDB51


## Move Dataset

In [ ]:
newPath = "/content/HMDB51"
# 左側 "files" 需開啟 "show hidden files" 的功能
if (os.path.exists(path) and path != newPath):
  !rm -rf /content/HMDB51
  !cp -r {path} /content
  print("Moved the datasets folder to /content.")

%cd /content

# Update folder path
if os.path.exists(newPath):
  path=newPath
print("Path to dataset files:", path)

Moved the datasets folder to /content.
/content
Path to dataset files: /content/HMDB51


## Ensuring Data Completeness via System Warm-up

In [ ]:
def warmup_filesystem(dataset_path):
    print(f"🚀 開始強制喚醒檔案系統快取：{dataset_path}")
    print("這可能需要 1~2 分鐘，請耐心等待系統掃描所有檔案...")

    start_time = time.time()
    total_files = 0
    total_dirs = 0

    # os.walk 會像爬蟲一樣，狠狠地鑽進每一個子資料夾
    # 這等同於你手動把每一個資料夾都點開來看！
    for root, dirs, files in os.walk(dataset_path):
        total_dirs += 1
        total_files += len(files)

    elapsed_time = time.time() - start_time
    print(f"✅ 暖身完成！共掃描了 {total_dirs} 個資料夾，{total_files} 個檔案。")
    print(f"⏱️ 耗時：{elapsed_time:.2f} 秒。現在作業系統已經完全清醒了！")

print("Done")

Done


In [ ]:
# 假設你的資料夾路徑為 './hmdb51_images' (請替換為 Kaggle 實際解壓縮路徑)
dataset_path = path
warmup_filesystem(dataset_path)

🚀 開始強制喚醒檔案系統快取：/content/HMDB51
這可能需要 1~2 分鐘，請耐心等待系統掃描所有檔案...
✅ 暖身完成！共掃描了 6818 個資料夾，129093 個檔案。
⏱️ 耗時：0.18 秒。現在作業系統已經完全清醒了！


## Setting Dataset For TSM

In [ ]:
class HMDB51_TSM_Dataset(Dataset):
    def __init__(self, root_dir, num_segments=8, is_train=True, transform=None):
        self.root_dir = root_dir # root_dir/動作/影片名稱/每一幀動作圖片.jpg
        self.num_segments = num_segments # 分割數量
        self.is_train = is_train
        self.transform = transform

        self.classes = sorted(os.listdir(root_dir)) # 排序資料夾以免每次順序不同
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)} # 儲存所有動作名稱與順序

        self.video_data = []

        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name) # cls_dir = root_dir/cls_name
            if not os.path.isdir(cls_dir):
                continue

            for video_name in os.listdir(cls_dir):
                video_path = os.path.join(cls_dir, video_name) # video_path = root_dir/cls_name/video_name
                if os.path.isdir(video_path):

                    image_files = sorted(glob.glob(os.path.join(video_path, '*.[jJ][pP]*[gG]'))) # 找出所有video_path底下的.jpg, .JPG, .jpeg檔路徑
                    num_frames = len(image_files) # 找到的數量

                    if num_frames > 0:
                        self.video_data.append({
                            'video_path': video_path,
                            'label': self.class_to_idx[cls_name],
                            'frame_paths': image_files,  # 預先存好這支影片所有圖片的路徑
                            'num_frames': num_frames
                        })

        print(f"✅ 索引建立完成！共載入 {len(self.video_data)} 支有效影片。")

    def __len__(self):
        return len(self.video_data)

    def _sample_indices(self, num_frames):
        if num_frames <= self.num_segments:
            return [i % num_frames for i in range(self.num_segments)]

        segment_duration = num_frames // self.num_segments
        indices = []
        for i in range(self.num_segments):
            if self.is_train:
                offset = random.randint(0, segment_duration - 1)
            else:
                offset = segment_duration // 2
            indices.append(i * segment_duration + offset)
        return indices

    def __getitem__(self, idx):
        # 1. 從我們預先建好的名冊中直接提取資料，不需要再 os.listdir 或 glob！
        data = self.video_data[idx]
        label = data['label']
        image_files = data['frame_paths']
        num_frames = data['num_frames']

        # 2. 決定要抽哪 8 幀
        frame_indices = self._sample_indices(num_frames)

        images = []
        for fi in frame_indices:
            # 3. 直接根據精確路徑讀取圖片 (這對作業系統來說是最輕量、最穩定的操作)
            img_path = image_files[fi]
            try:
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                images.append(img)
            except Exception as e:
                # 萬一真的有某張圖片損壞，印出具體路徑並隨便塞個黑畫面避免訓練中斷
                print(f"⚠️ 讀取圖片失敗 {img_path}: {e}")
                images.append(torch.zeros((3, 224, 224)))

        clip_tensor = torch.stack(images, dim=0)
        return clip_tensor, label

print("Done")

Done


In [ ]:
# ImageNet 的標準均值與標準差
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# 訓練集的 Transform：加入隨機裁切與水平翻轉增加泛化能力
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize
])

# 測試/驗證集的 Transform：單純置中裁切
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize
])

# 實例化 Dataset
# 註：實務上你需要把 HMDB51 依照官方的 train/test split 拆成兩個資料夾，或在類別內用清單過濾
train_dataset = HMDB51_TSM_Dataset(root_dir=dataset_path, num_segments=8, is_train=True, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)

# 測試看看產出的資料維度
for clips, labels in train_loader:
  # 預期維度: [Batch_Size, T, C, H, W] -> [16, 8, 3, 224, 224]
  print(f"Clip tensor shape: {clips.shape}")
  print(f"Labels shape: {labels.shape}")
  break

✅ 索引建立完成！共載入 6750 支有效影片。


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Clip tensor shape: torch.Size([16, 8, 3, 224, 224])
Labels shape: torch.Size([16])


# **Model Construction**

In [ ]:
class TemporalShift(nn.Module):
    def __init__(self, net, n_segment=8, n_div=8):
        """
        Args:
            net (nn.Module): 原本的 2D 卷積層或模組
            n_segment (int): 我們切的幀數 (T=8)
            n_div (int): 決定要平移多少比例的通道。設為 8 代表左推 1/8，右推 1/8。
        """
        super(TemporalShift, self).__init__()
        self.net = net
        self.n_segment = n_segment
        self.fold = n_div

    def forward(self, x):
        # x 進來的形狀會是 [B*T, C, H, W]，因為 2D ResNet 只看得懂 4 維
        nt, c, h, w = x.size()
        n_batch = nt // self.n_segment

        # 1. 變形回 [B, T, C, H, W] 才能做時間維度的位移
        x = x.view(n_batch, self.n_segment, c, h, w)

        # 準備一個空的 Tensor 來裝平移後的結果
        out = torch.zeros_like(x)

        # 算出 1/8 的通道數是多少
        fold = c // self.fold

        # 2. 開始魔法位移！
        # 通道 0 ~ fold：將未來的幀往回拉 (Shift left)
        out[:, :-1, :fold] = x[:, 1:, :fold]

        # 通道 fold ~ 2*fold：將過去的幀往未來推 (Shift right)
        out[:, 1:, fold: 2 * fold] = x[:, :-1, fold: 2 * fold]

        # 通道 2*fold ~ 結尾：留在原地不動
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]

        # 3. 變形回 [B*T, C, H, W] 交給原本的 2D 卷積層
        out = out.view(nt, c, h, w)
        return self.net(out)

print("Done")

Done


In [ ]:
class CustomTSMNet(nn.Module):
    def __init__(self, num_classes=51, n_segment=8):
        super().__init__()
        self.n_segment = n_segment

        # ==========================================
        # 第 1 層：單純提取空間特徵 (圖片剛進來，先不做時間平移)
        # ==========================================
        # 輸入: 3 通道 (RGB), 輸出: 32 種特徵圖, kernel: 5。用 stride=2 讓長寬減半
        self.conv1 = self._make_conv_block(3, 32, 5, 2, use_tsm=False)
        # 輸入: 32 通道, 輸出: 48 種特徵圖, kernel: 3。用 stride=2 讓長寬減半
        self.conv2 = self._make_conv_block(32, 48, 3, 1)
        # 輸入: 48 通道, 輸出: 48 種特徵圖, kernel: 3。用 stride=1
        self.conv3 = self._make_conv_block(48, 48, 3, 1)
        # 輸入: 48 通道, 輸出: 48 種特徵圖, kernel: 1。用 stride=1
        self.conv4 = self._make_conv_block(48, 48, 1, 1, use_tsm=False)

        # 輸入: 96 通道, 輸出: 64 種特徵圖, kernel: 3。用 stride=1
        self.conv5 = self._make_conv_block(96, 64, 3, 1)
        # 輸入: 64 通道, 輸出: 64 種特徵圖, kernel: 3。用 stride=1
        self.conv6 = self._make_conv_block(64, 64, 3, 1)
        # 輸入: 64 通道, 輸出: 64 種特徵圖, kernel: 3。用 stride=1
        self.conv7 = self._make_conv_block(64, 64, 1, 1, use_tsm=False)

        # 輸入: 128 通道, 輸出: 128 種特徵圖, kernel: 3。用 stride=1
        self.conv8 = self._make_conv_block(128, 128, 3, 1)
        # 輸入: 128 通道, 輸出: 128 種特徵圖, kernel: 3。用 stride=1
        self.conv9 = self._make_conv_block(128, 128, 3, 1)
        # 輸入: 128 通道, 輸出: 128 種特徵圖, kernel: 1。用 stride=1
        self.conv10 = self._make_conv_block(128, 128, 1, 1, use_tsm=False)

        # ==========================================
        # 分類器 (Fully Connected Layer)
        # ==========================================
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5), # 防止自己建的模型過擬合死背
            nn.Linear(in_features=256, out_features=num_classes)
        )

    def _make_conv_block(self, in_c, out_c, k, s, p=0, use_tsm=True):
      '''
      in_c: in_channels
      out_c: out_channels
      k: kernel_size
      s: stride
      p: padding
      '''
      conv_layer = nn.Conv2d(in_channels=in_c, out_channels=out_c, kernel_size=k, stride=s, padding=p)

      # 如果開啟 use_tsm，就把 conv_layer 用 TemporalShift 包起來
      if use_tsm:
          conv_layer = TemporalShift(conv_layer, n_segment=self.n_segment, n_div=8)

      return nn.Sequential(
          conv_layer,
          nn.BatchNorm2d(out_c),
          nn.ReLU(),
      )

    def forward(self, x):
        # 訓練迴圈送進來的 x 維度是 [128, 3, 224, 224] (已經被我們壓成 B*T 了)
        out1 = self.conv1(x)
        out2 = self.conv2(out1)
        out3 = self.conv3(out2)
        out4 = self.conv4(out3)

        out_concat = torch.cat((out3, out4), dim=1)
        out_pool = nn.MaxPool2d(kernel_size=2, stride=2)(out_concat)

        out5 = self.conv5(out_pool)
        out6 = self.conv6(out5)
        out7 = self.conv7(out6)

        out_concat = torch.cat((out6, out7), dim=1)
        out_pool = nn.MaxPool2d(kernel_size=2, stride=2)(out_concat)

        out8 = self.conv8(out_pool)
        out9 = self.conv9(out8)
        out10 = self.conv10(out9)

        out_concat = torch.cat((out9, out10), dim=1)
        out_pool = nn.MaxPool2d(kernel_size=2, stride=2)(out_concat)

        out_pool = nn.AdaptiveAvgPool2d((1, 1))(out_pool)

        # 到這裡 out_pool 的維度是 [128, 256, 1, 1]，無法直接餵給 Linear，必須攤平
        flat_x = out_pool.view(out_pool.size(0), -1) # 攤平成 [128, 256]

        # 進行最終的 51 類預測
        out = self.classifier(flat_x) # 輸出維度: [128, 51]

        return out

print("Done")

Done


# **Model Instantiation**

In [ ]:
# 建立你親自打造的模型！
custom_model = CustomTSMNet(num_classes=51, n_segment=8)

# 載入訓練過的權重 若無 weight_path可直接留空
# (如果有使用 GPU 訓練，但要在 CPU 電腦上讀取，可以加上 map_location=torch.device('cpu'))

weights_num = 95
weights_acc = 72.36

weights_path = f"/content/drive/MyDrive/HARCNN/best_tsm_hmdb51_{weights_num}_acc_{weights_acc:.2f}.pth"
if weights_path:
  if os.path.exists(weights_path):
    weights = torch.load(weights_path)
    custom_model.load_state_dict(weights)
  else:
    print(f"未找到路徑 {weights_path}")
print("Done！")

Done！


# **Training Model**

In [ ]:
# ==========================================
# 1. 訓練前置設定
# ==========================================
# 檢查是否有 GPU 可以用 (強烈建議在 Kaggle 打開 GPU P100 或 T4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"目前使用的運算設備: {device}")

# 把我們做好的模型搬到 GPU 上
custom_model = custom_model.to(device)

# 設定損失函數 (多分類問題標準配備)
criterion = nn.CrossEntropyLoss()

# 設定優化器 (論文中提到 SGD 搭配適當的 Learning Rate 效果很好)
optimizer = optim.SGD(custom_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

scheduler = lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40, 45], gamma=0.1)

# 設定訓練週期
num_epochs = 50
best_val_acc = weights_acc

# ==========================================
# 2. 開始訓練迴圈
# ==========================================
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 10)

    # --- 訓練階段 ---
    custom_model.train() # 告訴模型現在是訓練模式 (會啟用 Dropout 和 Batch Norm 的更新)
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in progress_bar:
        # 1. 把資料搬到 GPU
        if (len(inputs) == 0 or len(labels) == 0):
          print("error")
          continue
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # ==========================================
        # 【關鍵修正 1：輸入前融合 Batch 與 Time 維度】
        # 原本 inputs: [16, 8, 3, 224, 224] -> [B, T, C, H, W]
        # ==========================================
        B, T, C, H, W = inputs.size()
        inputs = inputs.view(B * T, C, H, W) # 變形為 [128, 3, 224, 224]

        # 3. 前向傳播 (Forward Pass)
        # 模型現在看的是 128 張圖片，所以 outputs 會是 [128, num_classes]
        outputs = custom_model(inputs)

        # ==========================================
        # 【關鍵修正 2：輸出後做時間維度平均 (Consensus)】
        # 將 [128, 51] 變回 [16, 8, 51]，然後在 T (第 1 維度) 上取平均
        # ==========================================
        outputs = outputs.view(B, T, -1).mean(dim=1) # 變形為 [16, 51]

        # 4. 計算誤差 (Loss)
        # 現在 outputs 是 [16, 51]，labels 是 [16]，維度完美對齊！
        loss = criterion(outputs, labels)

        # 5. 反向傳播
        loss.backward()

        # 6. 更新參數
        optimizer.step()

        # 統計準確率與 Loss
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total * 100
    progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})
    print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%")


    # ==========================================
    # 3. 驗證階段 (如果有設定 val_loader)
    # ==========================================
    # 假設你已經按照 train_loader 的邏輯，做了一個 val_loader
    custom_model.eval() # 切換到評估模式
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad(): # 評估時不需要計算梯度，可以省下大量記憶體與時間
        progress_bar = tqdm(train_loader, desc="Validing", leave=False) # 註：這裡建議換成 val_loader
        for inputs, labels in progress_bar:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # ==========================================
            # 🚨 必須補上！跟訓練時一樣，先把 Batch 和 Time 壓扁
            # ==========================================
            B, T, C, H, W = inputs.size()
            inputs = inputs.view(B * T, C, H, W)

            # 模型進行預測
            outputs = custom_model(inputs)

            # ==========================================
            # 🚨 必須補上！輸出後做時間維度平均
            # ==========================================
            outputs = outputs.view(B, T, -1).mean(dim=1)

            loss = criterion(outputs, labels)

            # --- 下面的統計計算記得也要改 ---
            # 因為我們做了 mean(dim=1)，現在 outputs 回到了 B (也就是 16)
            # 所以計算 loss 權重時，要用 B (而不是 inputs.size(0)，因為那時候 inputs 是 128)
            val_loss += loss.item() * B
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            # 即時更新進度條的 Loss
            progress_bar.set_postfix({'Val Loss': f"{loss.item():.4f}"})

    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total * 100
    print(f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:.2f}%")

    if val_epoch_acc > best_val_acc:
        print(f"🔥 發現更好的模型！準確率從 {best_val_acc:.2f}% 提升至 {val_epoch_acc:.2f}%")
        best_val_acc = val_epoch_acc

        # 立刻把這個最棒的模型存下來！
        save_path = f'/content/drive/MyDrive/HARCNN/best_tsm_hmdb51_{weights_num + 1 + epoch}_acc_{val_epoch_acc:.2f}.pth'
        torch.save(custom_model.state_dict(), save_path)
        print("💾 最佳模型已儲存！")

    # 在印出 validation 準確率之後，讓排程器更新一次學習率
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    print(f"目前學習率: {current_lr:.6f}")
    print("-" * 20 + "\n")

print("訓練完成！")

目前使用的運算設備: cuda
Epoch 1/50
----------


Train Loss: 14.6401 | Train Acc: 47.53%


Val Loss: 1.4830 | Val Acc: 56.77%
目前學習率: 0.010000
--------------------

Epoch 2/50
----------


Train Loss: 13.8802 | Train Acc: 49.81%


Val Loss: 1.4221 | Val Acc: 58.21%
目前學習率: 0.010000
--------------------

Epoch 3/50
----------


...

Val Loss: 1.3801 | Val Acc: 59.69%
目前學習率: 0.010000
--------------------

Epoch 5/50
----------


Train Loss: 13.1681 | Train Acc: 52.81%


Val Loss: 1.4204 | Val Acc: 58.33%
目前學習率: 0.010000
--------------------

Epoch 6/50
----------


Train Loss: 13.2078 | Train Acc: 52.06%


Validing:  47%|████▋     | 197/422 [01:38<01:37,  2.31it/s, Val Loss=1.5800]